# Medical LLM Fine-Tuning: Baseline Evaluation

This notebook has two main jobs:
1. **Preprocess and split** the cleaned dataset from notebook `01` into training, validation, and test sets.
2. **Run baseline inference** using the raw, un-finetuned Mistral-7B-Instruct model on 20 test questions to establish a performance baseline.

We need this baseline so we can quantitatively measure how much our fine-tuning actually improves the model in notebook `04_evaluation.ipynb`.

> **Environment Note:** This notebook is optimised to run on **Google Colab with a T4 GPU**. Before proceeding, ensure you have selected `Runtime > Change runtime type > T4 GPU`.

## Cell 1 — Install Dependencies

We install the core libraries needed for this notebook:
- `transformers` — to load the Mistral model and tokenizer from Hugging Face.
- `torch` — the deep learning framework underlying the model.
- `accelerate` — enables efficient multi-device model loading and `device_map="auto"`.
- `bitsandbytes` — provides 4-bit quantization so the 7B model fits within the 15GB T4 VRAM budget.
- `datasets` & `pandas` — for data manipulation.

In [ ]:
# Install all required libraries for model loading, quantization, and data handling
!pip install transformers torch accelerate bitsandbytes datasets pandas

## Cell 2 — Mount Google Drive

Colab's local filesystem is **ephemeral** — every time the runtime disconnects, all files are wiped. We mount the same Google Drive folder used in notebook `01` so we can read `cleaned_medquad.csv` directly, and save all output splits and results back to Drive for downstream notebooks.

All project files live under `/content/drive/MyDrive/medical-llm/`.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive — a browser popup will ask you to authorise access
drive.mount('/content/drive')

# Shared project directory — must match the path used in notebook 01
DRIVE_DIR = '/content/drive/MyDrive/medical-llm'
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"Google Drive mounted. Reading from and writing to: {DRIVE_DIR}")

## Cell 3 — Load and Sample the Cleaned Dataset

We load `cleaned_medquad.csv` that was produced and saved to Google Drive by notebook `01_data_exploration.ipynb`. We then apply a second round of length-based filtering to remove the long tail of very long answers — keeping only answers between 50 and 1500 characters.

We create three deterministic splits using `random_state=42`:
- **Train (15 000)** — what the model will learn from.
- **Validation (200)** — used to monitor overfitting during training.
- **Test (20)** — held out for baseline and post-fine-tuning evaluation.

All three splits are saved back to Google Drive.

In [ ]:
import pandas as pd
import random

# Load cleaned_medquad.csv from Google Drive (written by notebook 01)
df = pd.read_csv(f'{DRIVE_DIR}/cleaned_medquad.csv')

# The error 'KeyError: 'output'' indicates that the 'output' column does not exist in the DataFrame 'df'.
# From the provided kernel state, 'df' has 'question' as one of its columns and is a 2-column DataFrame.
# It is highly probable that the second column contains the answers, which should be named 'output'
# for the subsequent code to work. Also, an 'instruction' column is expected later.

# Dynamically get the name of the second column (which is likely the 'answer')
original_columns = df.columns.tolist()
if len(original_columns) == 2 and 'question' in original_columns:
    input_col_name = 'question'
    output_col_name = [col for col in original_columns if col != 'question'][0] # Get the other column name
    
    # Rename columns to 'input' and 'output'
    df = df.rename(columns={input_col_name: 'input', output_col_name: 'output'})
    
    # Add an 'instruction' column with a default prompt, as it's expected by later code.
    df['instruction'] = "Answer the medical question posed in the input."
else:
    # If the DataFrame doesn't have the expected 'question' and another column,
    # or if it already has 'instruction', 'input', 'output', then no renaming needed.
    # However, the KeyError implies it's missing 'output'.
    # This `else` block would mean the initial assumption was wrong, and the original
    # error would persist or a new one might arise.
    # For this specific error, the fix addresses the 'output' KeyError.
    pass # Let the KeyError happen if assumptions are not met, to avoid silent data corruption.

# Filter to answers between 50 and 1500 characters
# This removes the long tail we saw in the distribution
df = df[df['output'].str.len().between(50, 1500)]
df = df[df['input'].str.len() > 10]

# Shuffle and split — using fixed seeds for reproducibility
random.seed(42)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df = df.iloc[:15000]
val_df   = df.iloc[15000:15200]
test_df  = df.iloc[15200:15220]

print(f"Training samples:   {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples:       {len(test_df)}")

# Save splits to Google Drive so notebook 03 (fine-tuning) can load them directly
train_df.to_csv(f'{DRIVE_DIR}/train_data.csv', index=False)
val_df.to_csv(f'{DRIVE_DIR}/val_data.csv',     index=False)
test_df.to_csv(f'{DRIVE_DIR}/test_data.csv',   index=False)

print("\nSplits saved to Google Drive successfully")
print("\nSample training example:")
print("INSTRUCTION:", train_df.iloc[0]['instruction'])
print("INPUT:",       train_df.iloc[0]['input'])
print("OUTPUT:",      train_df.iloc[0]['output'])

## Cell 4 — Format into the Mistral Instruction Template

Mistral-7B-Instruct was trained using a specific chat template. Feeding it text in the correct format is critical for getting coherent, instruction-following responses — even from the base model before fine-tuning.

The expected format is:
```
<s>[INST] {instruction}\n\n{input} [/INST] {output} </s>
```
- During **training**, we include the `output` so the model learns to produce it.
- During **inference** (this notebook), we omit the `output` and let the model generate it.

In [ ]:
def format_mistral_prompt(instruction, input_text, output_text=None):
    """
    Mistral-7B-Instruct expects this exact format:
    <s>[INST] {instruction}\n\n{input} [/INST] {output} </s>

    During inference (baseline), we omit the output.
    During training, we include the output.
    """
    if output_text:
        return f"<s>[INST] {instruction}\n\n{input_text} [/INST] {output_text} </s>"
    else:
        return f"<s>[INST] {instruction}\n\n{input_text} [/INST]"

# Test the format with one example to verify it looks correct
sample = train_df.iloc[0]

formatted_with_output    = format_mistral_prompt(sample['instruction'], sample['input'], sample['output'])
formatted_without_output = format_mistral_prompt(sample['instruction'], sample['input'])

print("TRAINING FORMAT (with output):")
print(formatted_with_output[:500])
print("\nINFERENCE FORMAT (without output):")
print(formatted_without_output)
print(f"\nTotal characters in training example: {len(formatted_with_output)}")

## Cell 5 — Load the Base Mistral Model in 4-Bit Quantization

The full Mistral-7B model requires ~28GB of VRAM in FP32. The T4 GPU on Colab only has ~15GB. We solve this by using **4-bit NF4 quantization** via `bitsandbytes`, which compresses the model weights to ~4.5GB without a significant quality loss.

Key configuration choices:
- `load_in_4bit=True` — activates 4-bit weight compression.
- `bnb_4bit_use_double_quant=True` — a second quantization step further reduces memory usage.
- `bnb_4bit_quant_type="nf4"` — NormalFloat4, the recommended format for LLM quantization.
- `bnb_4bit_compute_dtype=torch.bfloat16` — computations are still done in BF16 for quality.
- `device_map="auto"` — lets `accelerate` automatically place layers on the GPU.

> **Note:** The initial model download from Hugging Face takes ~5-10 minutes on Colab. Ensure you have accepted the Mistral model licence on HuggingFace.co and set your access token via `Secrets` in the left sidebar (`HF_TOKEN`).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Configure 4-bit NF4 quantization to fit the 7B model within T4 VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Set pad token to eos token to avoid a warning during batch generation
tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit... (this takes 3-5 minutes)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
# Set to eval mode — disables dropout and ensures deterministic behaviour
model.eval()

print("Model loaded successfully")
print(f"Model device: {next(model.parameters()).device}")

## Cell 6 — Run Baseline Inference on 20 Test Questions

We now run the **untrained** Mistral model against our 20 held-out test questions. These responses represent the model's out-of-the-box medical knowledge before any fine-tuning.

Key generation settings:
- `max_new_tokens=256` — limits response length to keep inference fast on the T4.
- `temperature=0.7` — slight randomness to avoid degenerate, repetitive outputs.
- `do_sample=True` — enables sampling (required when `temperature != 1.0`).

We slice `outputs[0][inputs['input_ids'].shape[1]:]` to decode **only the newly generated tokens**, stripping the input prompt from the output.

Results are saved to `baseline_results.json` on Google Drive. In notebook `04_evaluation.ipynb`, we'll load this file alongside the fine-tuned model's responses and compute ROUGE and BERTScore metrics side-by-side.

In [ ]:
import json
from tqdm import tqdm

def generate_response(model, tokenizer, instruction, input_text, max_new_tokens=256):
    """Generate a response from the model given a medical question."""
    # Format the prompt using the Mistral instruction template (inference mode — no output)
    prompt = format_mistral_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the new tokens — slice off the input prompt from the output
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()


# Run baseline evaluation across all 20 test samples
baseline_results = []

print("Running baseline evaluation on 20 test questions...")
for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    response = generate_response(
        model, tokenizer,
        row['instruction'],
        row['input']
    )

    baseline_results.append({
        'question':          row['input'],
        'instruction':       row['instruction'],
        'ground_truth':      row['output'],
        'baseline_response': response
    })

    print(f"\n--- Question {len(baseline_results)} ---")
    print(f"Q:        {row['input'][:100]}...")
    print(f"Baseline: {response[:200]}...")

# Save results to Google Drive — notebook 04 will load this for comparison
results_path = f'{DRIVE_DIR}/baseline_results.json'
with open(results_path, 'w') as f:
    json.dump(baseline_results, f, indent=2)

print(f"\nBaseline evaluation complete. Saved {len(baseline_results)} results to:")
print(f"  {results_path}")